# **Day 64: Closures**

A **closure** is an **inner function** that **remembers** the variables of the outer function, even after the outer function has finished.

**Real-life example:** A **tiffin box** packed by your mother. Even after you leave home, the box still has the food from home inside it. The inner function "carries" the outer function's variables with it.

In [1]:
def make_greeter(greeting):
    def greet(name):
        return f"{greeting}, {name}!"      # Uses 'greeting' from the outer function
    return greet

bengali = make_greeter("Nomoskar")
english = make_greeter("Hello")

print(bengali("Tanmoy"))
print(english("Tanmoy"))

Nomoskar, Tanmoy!
Hello, Tanmoy!


`make_greeter` has finished running, but `bengali` still remembers that `greeting` was "Nomoskar". That remembered value is stored inside the closure.

In [2]:
print(bengali.__closure__[0].cell_contents)

Nomoskar


## **Closures with Changing Data (nonlocal)**
To **change** a remembered variable, we use `nonlocal`.

In [3]:
def make_counter(start=0):
    count = start
    def increase(step=1):
        nonlocal count
        count += step
        return count
    return increase

token = make_counter(100)
print(token(), token(), token())

another = make_counter()
print(another(), token())        # Each closure has its own separate count

101 102 103
1 104


## **Example from Remote Sensing: A Scaling Function Factory**
Every satellite product stores its pixel values with a **scale factor** and an **offset**. To get the real value, we use:

**real value = stored value x scale + offset**

Instead of passing the scale and offset every time, we can create a **ready-made converter** for each product using a closure.

In [4]:
def make_scaler(scale, offset=0.0):
    def convert(dn):
        return round(dn * scale + offset, 4)
    return convert

landsat_sr = make_scaler(0.0000275, -0.2)      # Landsat Collection 2 surface reflectance
sentinel_sr = make_scaler(0.0001)              # Sentinel-2 surface reflectance
modis_lst = make_scaler(0.02)                  # MODIS land surface temperature (in Kelvin)

print("Landsat  DN 12000 ->", landsat_sr(12000))
print("Sentinel DN  3500 ->", sentinel_sr(3500))

kelvin = modis_lst(15200)
print("MODIS    DN 15200 ->", kelvin, "K =", round(kelvin - 273.15, 2), "C")

Landsat  DN 12000 -> 0.13
Sentinel DN  3500 -> 0.35
MODIS    DN 15200 -> 304.0 K = 30.85 C


- Each converter **remembers** its own scale and offset.
- Now the rest of the program just calls `landsat_sr(dn)` without worrying about the numbers.

## **A Common Mistake: Late Binding in Loops**

In [5]:
funcs = []
for i in range(3):
    funcs.append(lambda: i)
print([f() for f in funcs])       # We expected [0, 1, 2]

[2, 2, 2]


All three lambdas look at `i` **when they are called**, and by then the loop has finished and `i` is 2.

**The fix:** Use a default argument. Default values are saved **when the function is made**.

In [6]:
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])

[0, 1, 2]


## **Closure vs Class**
A closure with one function is like a small object with one method. For **one** action, a closure is simple and neat. For **many** actions, a class is clearer.

In [7]:
def running_stats():
    n = total = 0
    low, high = float("inf"), float("-inf")
    def add(x):
        nonlocal n, total, low, high
        n += 1
        total += x
        low, high = min(low, x), max(high, x)
        return {"count": n, "mean": round(total / n, 2), "min": low, "max": high}
    return add

stats = running_stats()
for aqi in [142, 188, 165, 210]:
    print(stats(aqi))

{'count': 1, 'mean': 142.0, 'min': 142, 'max': 142}
{'count': 2, 'mean': 165.0, 'min': 142, 'max': 188}
{'count': 3, 'mean': 165.0, 'min': 142, 'max': 188}
{'count': 4, 'mean': 176.25, 'min': 142, 'max': 210}


## **Practice Problem: Joining Functions Together**

In [8]:
from functools import reduce

def compose(*functions):
    return lambda x: reduce(lambda acc, f: f(acc), functions, x)

make_file_name = compose(str.strip, str.lower, lambda s: s.replace(" ", "_"), lambda s: s + ".tif")
print(make_file_name("  Paschim Medinipur NDVI  "))

paschim_medinipur_ndvi.tif


In [9]:
# Quick question
# Guess the output of this code cell
# def outer():
#     x = 1
#     def inner():
#         return x + 1
#     return inner
# print(outer()())